# Compute Article Embeddings (run this notebook on Kaggle)

This notebook is **not** part of the local one-command pipeline -- it needs a
GPU, so it's meant to be run standalone on Kaggle, with the results
downloaded back into the local repo. See `SPEC.md` Q3 section 1 for why
(GPU-backed compute for the embedding model itself; the local repo only ever
does inference/retrieval math on the resulting vectors, no ML library needed
there).

Model: `paraphrase-xlm-r-multilingual-v1` (XLM-RoBERTa-based, 768-dim,
handles Danish and English in one model -- matches the assignment's
"BERT/XLM-RoBERTa" suggestion for Q3).

To compute embeddings for `ebnerd_large`/`mind_large`: create a Kaggle
Dataset containing `data/processed/ebnerd_large/articles.parquet` and
`data/processed/mind_large/articles.parquet` (rename each to
`ebnerd_large_articles.parquet` / `mind_large_articles.parquet` on upload so
the discovery patterns below match them), attach it via "+ Add Data", enable
Settings > Internet > On and Settings > Accelerator > GPU T4, then
Save & Run All. Download `ebnerd_large_article_embeddings.parquet` and
`mind_large_article_embeddings.parquet` from the resulting version's Output
tab and place them at `data/processed/ebnerd_large/article_embeddings.parquet`
and `data/processed/mind_large/article_embeddings.parquet` locally.

In [ ]:
!pip install -q sentence-transformers

## Locate the uploaded `articles.parquet` files

Auto-discovers whichever of `ebnerd`/`ebnerd_small`/`ebnerd_large`/`mind`/
`mind_large` articles files are present under `/kaggle/input/` by filename
pattern -- you don't need to upload all five every time, only the ones you
actually need embeddings for (e.g. just `ebnerd_large_articles.parquet` and
`mind_large_articles.parquet` for this round). Longer/more-specific prefixes
(`ebnerd_small`, `ebnerd_large`, `mind_large`) are matched and excluded from
their shorter counterparts (`ebnerd`, `mind`) before those broader patterns
run, so e.g. `mind_large_articles.parquet` is never ambiguously counted as a
plain `mind` match. If discovery fails, check the exact path shown in the
Kaggle "Data" sidebar for your attached dataset.

In [ ]:
from glob import glob

import pandas as pd

# Order matters: each "_small"/"_large"/"_test_new" variant must be matched
# (and excluded from its plainer counterpart) before that plainer pattern
# runs, since e.g. "mind_large_test_new_articles.parquet" would otherwise
# also satisfy "*mind_large*articles*" and "*mind*articles*".
DATASET_PATTERNS = {
    "ebnerd_small": "/kaggle/input/**/*ebnerd_small*articles*.parquet",
    "ebnerd_large": "/kaggle/input/**/*ebnerd_large*articles*.parquet",
    "ebnerd": "/kaggle/input/**/*ebnerd*articles*.parquet",
    "mind_large_test_new": "/kaggle/input/**/*mind_large_test_new*articles*.parquet",
    "mind_large": "/kaggle/input/**/*mind_large*articles*.parquet",
    "mind": "/kaggle/input/**/*mind*articles*.parquet",
}


def find_articles_files() -> dict[str, str]:
    found = {}
    claimed = set()
    for name, pattern in DATASET_PATTERNS.items():
        matches = [m for m in glob(pattern, recursive=True) if m not in claimed]
        if len(matches) > 1:
            raise FileNotFoundError(f"expected at most one match for {name} ({pattern!r}), found {matches}")
        if len(matches) == 1:
            found[name] = matches[0]
            claimed.add(matches[0])
    if not found:
        raise FileNotFoundError(
            "no *articles*.parquet files found under /kaggle/input/ -- check the Data sidebar for the real path"
        )
    return found


articles_paths = find_articles_files()
print("found:", articles_paths)

articles_by_dataset = {name: pd.read_parquet(path) for name, path in articles_paths.items()}
{name: df.shape for name, df in articles_by_dataset.items()}

In [ ]:
EXPECTED_PREFIXES = {
    "ebnerd_small": "ebnerd_small_",
    "ebnerd_large": "ebnerd_large_",
    "ebnerd": "ebnerd_",
    "mind_large_test_new": "mind_large_test_",
    "mind_large": "mind_large_",
    "mind": "mind_",
}

for name, articles in articles_by_dataset.items():
    assert {"article_id", "title", "abstract"}.issubset(articles.columns)
    assert articles["article_id"].str.startswith(EXPECTED_PREFIXES[name]).all()
print("ok: uploaded files match the expected unified schema (SPEC.md Q1 section 2)")

## Load the multilingual model onto GPU

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("WARNING: no GPU detected -- Settings > Accelerator > GPU for reasonable speed.")

MODEL_NAME = "paraphrase-xlm-r-multilingual-v1"
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
EMBEDDING_DIM = model.get_sentence_embedding_dimension()
print("device:", DEVICE, "| embedding dim:", EMBEDDING_DIM)

In [ ]:
assert EMBEDDING_DIM > 0
print("ok: multilingual model loaded")

## Encode `title + abstract` for each uploaded dataset

Same corpus-text convention as Q2/Q3 (`title + " " + abstract`, `abstract`
`.fillna("")`'d -- see `SPEC.md` Q2 section 3 for why the fillna matters).

In [ ]:
def build_corpus_text(articles: pd.DataFrame) -> list[str]:
    return (articles["title"].fillna("") + " " + articles["abstract"].fillna("")).tolist()


embeddings_by_dataset = {}
for name, articles in articles_by_dataset.items():
    texts = build_corpus_text(articles)
    embeddings_by_dataset[name] = model.encode(texts, batch_size=256, show_progress_bar=True, convert_to_numpy=True)

{name: emb.shape for name, emb in embeddings_by_dataset.items()}

In [ ]:
import numpy as np

for name, articles in articles_by_dataset.items():
    emb = embeddings_by_dataset[name]
    assert emb.shape == (len(articles), EMBEDDING_DIM)
    assert not np.isnan(emb).any()
print("ok: embeddings computed for every uploaded dataset, correct shape, no NaNs")

## Save to `/kaggle/working/`

Output schema matches `SPEC.md` Q3 section 6 (`article_id, dataset,
embedding`). Anything written to `/kaggle/working/` becomes downloadable
from this version's **Output** tab after Save & Run All completes. Each
file is named `{dataset}_article_embeddings.parquet` (e.g.
`ebnerd_small_article_embeddings.parquet`); rename to `article_embeddings.parquet`
when placing it at `data/processed/{dataset}/` locally.

In [ ]:
def save_embeddings(articles: pd.DataFrame, embeddings: np.ndarray, dataset: str, out_path: str) -> str:
    df = pd.DataFrame({
        "article_id": articles["article_id"].to_numpy(),
        "dataset": dataset,
        "embedding": [row.astype("float32").tolist() for row in embeddings],
    })
    df.to_parquet(out_path, index=False)
    return out_path


output_paths = {
    name: save_embeddings(
        articles_by_dataset[name], embeddings_by_dataset[name], name,
        f"/kaggle/working/{name}_article_embeddings.parquet",
    )
    for name in articles_by_dataset
}
output_paths

# Manual Review Complete